Improvements to reduce overfitting:
- L2 regularisation in the loss function (penalising large weights)
- lowering the learning rate
- freeze most of GPT-2 -> only train the last few blocks

In [1]:
import json
import os
from PIL import Image
from datasets import Dataset, DatasetDict

dataset_dir = "VQA-RAD_dataset"
image_dir   = os.path.join(dataset_dir, "VQA_RAD Image Folder")

with open(os.path.join(dataset_dir, "VQA_RAD Dataset Public.json")) as f:
    records = json.load(f)

# ── Clean, canonical VQA-RAD split (same as the BLIP-2 notebooks) ─────────────
# The old version expanded every record with BOTH its question_rephrase and
# question_frame fields. But paraphrases already exist as their own "para"
# records, so re-adding question_rephrase DUPLICATES them (~39% dup rows in
# train, ~34% in test) and inflates the test set to 1,087. Fixes:
#   (1) do NOT expand question_rephrase (paraphrases are already separate rows);
#   (2) add the templated "framed" questions (which exist ONLY as a field) to
#       TRAIN only -- adding a test question's framing to train would leak test;
#   (3) drop exact (image, question, answer) duplicates.
# Result: 2,393 train / 451 canonical test, leakage-safe and comparable.
def load_records(recs, add_framed):
    rows = {"image_path": [], "question": [], "answer": [],
            "question_type": [], "answer_type": [], "image_organ": []}
    seen = set()

    def add_row(img_path, question, rec):
        key = (img_path,
               " ".join(str(question).strip().lower().split()),
               str(rec["answer"]).strip().lower())
        if key in seen:                      # skip exact-duplicate (image, question, answer)
            return
        seen.add(key)
        rows["image_path"].append(img_path)
        rows["question"].append(str(question))
        rows["answer"].append(str(rec["answer"]))
        rows["question_type"].append(str(rec.get("question_type", "OTHER")))
        rows["answer_type"].append(str(rec.get("answer_type", "OTHER")).strip().upper())
        rows["image_organ"].append(str(rec.get("image_organ", "")))

    for rec in recs:
        img_path = os.path.join(image_dir, rec["image_name"])
        if not os.path.isfile(img_path):
            continue
        add_row(img_path, rec["question"], rec)          # the record's own question
        if add_framed:                                   # templated framing -> TRAIN only
            frame = rec.get("question_frame", "NULL")
            if frame and frame != "NULL":
                add_row(img_path, frame, rec)
    return Dataset.from_dict(rows)

dataset = DatasetDict({
    "train": load_records([r for r in records if not r["phrase_type"].startswith("test_")], add_framed=True),
    "test":  load_records([r for r in records if     r["phrase_type"].startswith("test_")], add_framed=False),
})

first_row = dataset["train"][0]
print("Dataset loaded (clean canonical split)!")
print(f"  Train : {len(dataset['train'])} examples")
print(f"  Test  : {len(dataset['test'])}  examples")
print(f"\nSample entry:")
print(f"  Question : {first_row['question']}")
print(f"  Answer   : {first_row['answer']}")
print(f"  Image    : {first_row['image_path']}")


/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded (clean canonical split)!
  Train : 2393 examples
  Test  : 451  examples

Sample entry:
  Question : Are regions of the brain infarcted?
  Answer   : Yes
  Image    : VQA-RAD_dataset/VQA_RAD Image Folder/synpic54610.jpg


In [2]:
from collections import Counter

print("Analyzing Training Dataset Balance...\n")

# 1. Extract all the answers from the training split
# We convert them to strings, strip whitespace, and make them lowercase 
# so that "Yes", "yes", and " yes " are all counted as the same thing.
all_answers = [str(ans).strip().lower() for ans in dataset['train']['answer']]

# 2. Count how many times each answer appears
answer_counts = Counter(all_answers)

# 3. Print the top 10 most common answers overall
print("--- Top 10 Most Common Answers ---")
for ans, count in answer_counts.most_common(10):
    print(f"'{ans}': {count} times")

# 4. Check the exact balance of Yes/No questions
yes_count = answer_counts.get('yes', 0)
no_count = answer_counts.get('no', 0)
total_binary = yes_count + no_count

print("\n--- Yes/No Question Balance ---")
if total_binary > 0:
    yes_pct = (yes_count / total_binary) * 100
    no_pct = (no_count / total_binary) * 100
    print(f"Total Yes/No Questions: {total_binary}")
    print(f"YES: {yes_count} ({yes_pct:.1f}%)")
    print(f"NO:  {no_count} ({no_pct:.1f}%)")
    
    # Give a warning if it is highly imbalanced
    if yes_pct > 70 or no_pct > 70:
        print("\n⚠️ WARNING: Your Yes/No classes are highly imbalanced!")
        print("The model might just memorize the most frequent answer instead of learning.")
else:
    print("No 'Yes' or 'No' answers found.")

Analyzing Training Dataset Balance...

--- Top 10 Most Common Answers ---
'no': 667 times
'yes': 638 times
'axial': 31 times
'right': 27 times
'left': 19 times
'pa': 14 times
'ct': 10 times
'pancreas': 10 times
'one': 10 times
'left kidney': 10 times

--- Yes/No Question Balance ---
Total Yes/No Questions: 1305
YES: 638 (48.9%)
NO:  667 (51.1%)


In [3]:
# We need to install the library Microsoft used to build BioMedCLIP
# %pip install open_clip_torch 

import torch
import open_clip

print("Downloading and loading BioMedCLIP...")

# Load the model and the image processor directly from the Hugging Face Hub via open_clip
model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
biomedclip_model, image_processor, _ = open_clip.create_model_and_transforms(model_name)

# BioMedCLIP is a contrastive model (it has both a vision part and a text part).
# we only want the vision part, so we extract the vision encoder
vision_encoder = biomedclip_model.visual

# Move it to the GPU if you are on Colab (or MPS on Mac, or CPU)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
vision_encoder = vision_encoder.to(device)
vision_encoder.eval() # Set it to evaluation mode (we won't re-train the vision encoder)

print("BioMedCLIP Vision Encoder loaded successfully!")

# Let's test it on the image from Step 1!
# 1. Preprocess the image (resizes it to 224x224 and normalizes colors)
sample_image = Image.open(first_row['image_path']).convert('RGB')
processed_image = image_processor(sample_image).unsqueeze(0).to(device)

# 2. Pass it through the Vision Encoder
with torch.no_grad():
    image_features = vision_encoder(processed_image)

print(f"\nShape of the extracted visual features: {image_features.shape}")


BioMedCLIP Vision Encoder loaded successfully!

Shape of the extracted visual features: torch.Size([1, 512])


In [4]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer

class BioMedVQA(nn.Module):
    def __init__(self, vision_encoder, text_model_name='gpt2'):
        super().__init__()
        
        # 1. The Vision Encoder
        self.vision_encoder = vision_encoder
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
            
        # 2. The Language Model (Lightweight Head)
        self.llm = GPT2LMHeadModel.from_pretrained(text_model_name)
        # Freeze all of GPT-2 except the last 2 transformer blocks and the LM head
        TRAINABLE_BLOCKS = {6, 7, 8, 9, 10, 11}

        for name, param in self.llm.named_parameters():
            # print(name)
            block_num = None
            for part in name.split('.'):
                if part.isdigit():
                    block_num = int(part)
                    break
            # Unfreeze: last 2 blocks + final layer norm + LM head
            is_trainable = (
                (block_num is not None and block_num in TRAINABLE_BLOCKS)
                or 'ln_f' in name
                or 'lm_head' in name
            )
            param.requires_grad = is_trainable

        self.tokenizer = GPT2Tokenizer.from_pretrained(text_model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # 3. The Translator
        self.translator = nn.Linear(512, self.llm.config.hidden_size)

    def forward(self, images, input_ids, attention_mask, labels=None):
        # Step A: Get image features
        with torch.no_grad():
            image_features = self.vision_encoder(images)
            
        # Step B: Translate the image and reshape it into a "word"
        translated_image = self.translator(image_features)
        translated_image = translated_image.unsqueeze(1)
        
        # Step C: Get text embeddings
        text_embeddings = self.llm.transformer.wte(input_ids)
        
        # Step D: Glue them together!
        combined_embeddings = torch.cat((translated_image, text_embeddings), dim=1)
        
        # Step E: Update the attention mask so the LLM looks at the image
        image_attention = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        combined_attention = torch.cat((image_attention, attention_mask), dim=1)
        
        if labels is not None:
            # Create a column of -100s for the image word
            image_labels = torch.full((labels.shape[0], 1), -100, device=labels.device)
            # Glue the -100 to the front of the text labels
            combined_labels = torch.cat((image_labels, labels), dim=1)
        else:
            combined_labels = None
            
        # Step F: Generate the output
        outputs = self.llm(
            inputs_embeds=combined_embeddings, 
            attention_mask=combined_attention,
            labels=combined_labels
            )
        
        return outputs
    
    # Add this inside class VQAModel(nn.Module):
    def generate(self, images, input_ids, attention_mask, max_new_tokens=10, **kwargs):
        """
        A custom generate function that handles gluing the image and text together
        before passing it to the inner LLM's generate function.
        """
        with torch.no_grad():
            # A. Get image embeddings
            image_features = self.vision_encoder(images)
            translated_image = self.translator(image_features).unsqueeze(1)
            
            # B. Get text embeddings
            text_embeddings = self.llm.transformer.wte(input_ids)
            
            # C. Glue them together
            combined_embeddings = torch.cat((translated_image, text_embeddings), dim=1)
            
            # D. Update attention mask for the image token
            image_attention = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
            combined_attention = torch.cat((image_attention, attention_mask), dim=1)
            
            # E. Pass everything to the LLM's built-in generate loop
            return self.llm.generate(
                inputs_embeds=combined_embeddings,
                attention_mask=combined_attention,
                max_new_tokens=max_new_tokens,
                **kwargs # Passes any extra arguments (like pad_token_id) safely
            )

# Test the architecture
vqa_model = BioMedVQA(vision_encoder).to(device)
print("VQA Architecture successfully built and moved to the GPU!")


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3785.40it/s]


VQA Architecture successfully built and moved to the GPU!


In [5]:
from torch.utils.data import Dataset, DataLoader

class VQARADDataset(Dataset):
    def __init__(self, hf_dataset, image_processor, tokenizer):
        self.dataset = hf_dataset
        self.image_processor = image_processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        
        # 1. Process the Image
        raw_image = Image.open(row['image_path']).convert("RGB")
        processed_image = self.image_processor(raw_image)
        
        # 2. Format the Text
        question = row['question']
        answer = str(row['answer'])
        
        # We separate the string into the "Prompt" and the "Full Text"
        prompt_text = f"Question: {question} Answer:"
        full_text = f"{prompt_text} {answer}{self.tokenizer.eos_token}"
        
        # 3. Tokenize the Text
        tokens = self.tokenizer(
            full_text, 
            padding="max_length", 
            max_length=64, 
            truncation=True, 
            return_tensors="pt"
        )
        
        input_ids = tokens.input_ids.squeeze(0)
        attention_mask = tokens.attention_mask.squeeze(0)
        
        # 4. Create the Causal Labels with Question Masking
        labels = input_ids.clone()
        
        # Rule A: Ignore the padding tokens
        labels[attention_mask == 0] = -100
        
        # Rule B: Ignore the question tokens!
        # First, we tokenize *just* the prompt to find out exactly how many tokens long it is
        prompt_tokens = self.tokenizer(prompt_text, return_tensors="pt").input_ids.squeeze(0)
        prompt_length = len(prompt_tokens)
        
        # Next, we set everything from the start of the sequence up to the end of the prompt to -100
        # (We use min() just in case a massive question got truncated by our max_length=64 limit)
        mask_length = min(prompt_length, 64)
        labels[:mask_length] = -100
        
        return {
            "images": processed_image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

print("Building the DataLoader engine...")

train_dataset = VQARADDataset(dataset['train'], image_processor, vqa_model.tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Let's test it and look at the exact numbers!
test_batch = next(iter(train_dataloader))
print(f"Batch Image Shape: {test_batch['images'].shape}")

# Print the first item in the batch to prove the masking worked
first_label = test_batch['labels'][0]
print("\nLook at the labels for the first example in the batch:")
print(first_label)


Building the DataLoader engine...
Batch Image Shape: torch.Size([4, 3, 224, 224])

Look at the labels for the first example in the batch:
tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,   645, 50256,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100])


In [6]:
from torch.optim import AdamW
from tqdm.auto import tqdm # This gives us a nice visual progress bar

# 1. The Optimizer
# We only want to train weights that are NOT frozen
trainable_params = [p for p in vqa_model.parameters() if p.requires_grad]
# 5e-5 (0.00005) is the standard learning rate for fine-tuning LLMs
optimizer = AdamW(trainable_params, lr=2e-5, weight_decay=0.01)

# 2. Put the model in training mode (turns on things like Dropout)
vqa_model.train()

# An "epoch" is one full pass through the entire dataset. we'll do 3.
epochs = 5
print(f"Starting training for {epochs} epochs...\n")

for epoch in range(epochs):
    total_loss = 0
    
    # Wrap our dataloader in tqdm to get a progress bar
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for batch in progress_bar:
        # Step A: Move the batch of data to the GPU
        images = batch['images'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Step B: Zero the gradients (wipe the slate clean)
        optimizer.zero_grad()
        
        # Step C: The Forward Pass
        # Pass the data into our model. Because we included labels, it calculates the loss!
        outputs = vqa_model(
            images=images, 
            input_ids=input_ids, 
            attention_mask=attention_mask,
            labels=labels
        )
        
        # Extract the calculated error
        loss = outputs.loss
        
        # Step D: The Backward Pass (Calculus magic!)
        loss.backward()
        
        # Step E: Take a step downhill to fix the weights
        optimizer.step()
        
        # Update our progress bar so we can watch the loss go down
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
        
    # Print the average error at the end of the epoch
    avg_loss = total_loss / len(train_dataloader)
    print(f"End of Epoch {epoch+1} | Average Loss: {avg_loss:.4f}\n")

print("Training Complete! You just built and trained a Medical VLM!")


Starting training for 5 epochs...



Epoch 1: 100%|██████████| 599/599 [00:56<00:00, 10.52it/s, loss=1.31] 


End of Epoch 1 | Average Loss: 3.0663



Epoch 2: 100%|██████████| 599/599 [00:56<00:00, 10.69it/s, loss=0.854]


End of Epoch 2 | Average Loss: 2.0538



...

End of Epoch 3 | Average Loss: 1.7392



Epoch 4: 100%|██████████| 599/599 [00:56<00:00, 10.69it/s, loss=1.89] 


End of Epoch 4 | Average Loss: 1.5087



Epoch 5: 100%|██████████| 599/599 [00:56<00:00, 10.64it/s, loss=0.76] 

End of Epoch 5 | Average Loss: 1.3127

Training Complete! You just built and trained a Medical VLM!


In [7]:
# ── Save the fine-tuned weights to disk ──────────────────────────────────────
# Only the TRAINED parts are saved: the translator (512->768 bridge) and GPT-2
# (self.llm, which holds the unfrozen blocks + LM head). The BioMedCLIP vision
# encoder is frozen and reloads deterministically from open_clip, so we skip it.
import os
SAVE_DIR  = "/home/matei/biomed_vqa_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, "biomed_vqa_final.pt")

torch.save(
    {"translator": vqa_model.translator.state_dict(),
     "llm":        vqa_model.llm.state_dict()},
    SAVE_PATH,
)
print("saved fine-tuned weights ->", SAVE_PATH)


saved fine-tuned weights -> /home/matei/biomed_vqa_checkpoints/biomed_vqa_final.pt


In [8]:
# ── (Optional) Reload the fine-tuned weights WITHOUT retraining ───────────────
# After a kernel restart: run setup -> data -> BioMedCLIP -> model-build cells to
# reconstruct `vqa_model`, then run THIS cell INSTEAD of the training loop to
# restore the trained weights from disk.
SAVE_PATH = "/home/matei/biomed_vqa_checkpoints/biomed_vqa_final.pt"
ckpt = torch.load(SAVE_PATH, map_location=device)
vqa_model.translator.load_state_dict(ckpt["translator"])
vqa_model.llm.load_state_dict(ckpt["llm"])
vqa_model.eval()
print("restored fine-tuned weights from", SAVE_PATH)


restored fine-tuned weights from /home/matei/biomed_vqa_checkpoints/biomed_vqa_final.pt


/tmp/ipykernel_122406/3761898621.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(SAVE_PATH, map_location=device)


In [9]:
import torch
import string
from collections import Counter
from tqdm.auto import tqdm

# ── Text normalisation (mirrors SQuAD evaluation script) ─────────────────────
def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def exact_match(pred, gt):
    return normalize(pred) == normalize(gt)

def token_recall(pred, gt):
    """Fraction of ground-truth word-tokens that appear in the prediction.
    Uses Counter (multiset) intersection to handle repeated words correctly."""
    pred_tokens = Counter(normalize(pred).split())
    gt_tokens   = Counter(normalize(gt).split())
    if not gt_tokens:
        return 0.0
    overlap = sum((pred_tokens & gt_tokens).values())
    return overlap / sum(gt_tokens.values())

# ── Single evaluation pass over one dataset split ────────────────────────────
def evaluate_split(split_name, hf_dataset):
    vqa_model.eval()
    device = next(vqa_model.parameters()).device

    closed_em, closed_n       = 0, 0
    open_em, open_rec, open_n = 0, 0.0, 0

    for i in tqdm(range(len(hf_dataset)), desc=f"Evaluating {split_name}"):
        row         = hf_dataset[i]
        true_answer = str(row['answer'])
        ans_type    = str(row['answer_type']).upper()

        img = Image.open(row['image_path']).convert('RGB')
        image_tensor = image_processor(img).unsqueeze(0).to(device)
        prompt       = f"Question: {row['question']} Answer:"
        tok          = vqa_model.tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            generated_ids = vqa_model.generate(
                images=image_tensor,
                input_ids=tok.input_ids,
                attention_mask=tok.attention_mask,
                max_new_tokens=20,
                pad_token_id=vqa_model.tokenizer.eos_token_id
            )
        prediction = vqa_model.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

        if ans_type == "CLOSED":
            closed_n += 1
            if exact_match(prediction, true_answer):
                closed_em += 1
        else:
            open_n += 1
            if exact_match(prediction, true_answer):
                open_em += 1
            open_rec += token_recall(prediction, true_answer)

    # ── Print results ─────────────────────────────────────────────────────────
    sep = "=" * 52
    print(f"\n{sep}")
    print(f"  RESULTS — {split_name.upper()} SET")
    print(f"{sep}")

    if closed_n:
        print(f"\nClosed-ended  ({closed_n} questions)")
        print(f"  Exact Match Accuracy : {closed_em}/{closed_n}  ({100*closed_em/closed_n:.2f}%)")

    if open_n:
        print(f"\nOpen-ended  ({open_n} questions)")
        print(f"  Exact Match Accuracy : {open_em}/{open_n}  ({100*open_em/open_n:.2f}%)")
        print(f"  Token Recall         : {open_rec/open_n:.4f}  ({100*open_rec/open_n:.2f}%)")

    total    = closed_n + open_n
    total_em = closed_em + open_em
    if total:
        print(f"\nOverall  ({total} questions)")
        print(f"  Exact Match Accuracy : {total_em}/{total}  ({100*total_em/total:.2f}%)")
    print()

# ── Run evaluation ────────────────────────────────────────────────────────────
# Test set  → generalisation performance (the number that matters)
# Train set → memorisation check; a large gap vs test = overfitting
evaluate_split("test",  dataset["test"])
evaluate_split("train", dataset["train"])


Evaluating test:   0%|          | 0/451 [00:00<?, ?it/s]

Evaluating test: 100%|██████████| 451/451 [00:16<00:00, 27.90it/s]



  RESULTS — TEST SET

Closed-ended  (272 questions)
  Exact Match Accuracy : 154/272  (56.62%)

Open-ended  (179 questions)
  Exact Match Accuracy : 32/179  (17.88%)
  Token Recall         : 0.2359  (23.59%)

Overall  (451 questions)
  Exact Match Accuracy : 186/451  (41.24%)



Evaluating train: 100%|██████████| 2393/2393 [01:27<00:00, 27.39it/s]


  RESULTS — TRAIN SET

Closed-ended  (1406 questions)
  Exact Match Accuracy : 1047/1406  (74.47%)

Open-ended  (987 questions)
  Exact Match Accuracy : 221/987  (22.39%)
  Token Recall         : 0.3033  (30.33%)

Overall  (2393 questions)
  Exact Match Accuracy : 1268/2393  (52.99%)

